# 🚀 Startup Success Prediction
### End-to-end ML pipeline: Data Import → EDA → Feature Engineering → Modeling → Evaluation
---

## 📦 Step 1: Install & Import Dependencies

In [ ]:
# Install required packages
!pip install kagglehub pandas numpy matplotlib seaborn scikit-learn xgboost imbalanced-learn plotly -q

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ML
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# Style
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 14, 'axes.labelsize': 12})

print('✅ All libraries imported successfully!')

## 🔑 Step 2: Download Dataset via kagglehub

> **No kaggle.json needed!** `kagglehub` will prompt you to log in with your Kaggle username & API key on first run.  
> Get your API key at: https://www.kaggle.com/settings → API → **Create New Token**

In [ ]:
import kagglehub

# Downloads dataset and returns local path (prompts login on first run)
path = kagglehub.dataset_download('arindam235/startup-investments-crunchbase')
print('✅ Dataset downloaded to:', path)

In [ ]:
# List downloaded files
import glob
files = glob.glob(path + '/**/*', recursive=True)
for f in files:
    print(f)

## 📂 Step 3: Load & Inspect Data

In [ ]:
# Find CSV automatically from the kagglehub download path
import os, glob
csv_files = glob.glob(os.path.join(path, '**/*.csv'), recursive=True)
print('Found CSV files:', csv_files)

df = pd.read_csv(csv_files[0], encoding='latin-1', low_memory=False)

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

## 🧹 Step 4: Data Cleaning & Target Engineering

In [ ]:
# ── Missing value overview ───────────────────────────────────────────────────
miss = df.isnull().mean().sort_values(ascending=False)
miss_pct = miss[miss > 0]

fig, ax = plt.subplots(figsize=(12, 5))
miss_pct.plot(kind='bar', ax=ax, color='coral', edgecolor='white')
ax.set_title('Missing Data (%) per Column', fontweight='bold')
ax.set_ylabel('Missing Fraction')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Drop columns with > 60% missing ──────────────────────────────────────────
threshold = 0.60
df = df.loc[:, df.isnull().mean() < threshold]
print(f'Columns after dropping high-NaN cols: {df.shape[1]}')

# ── Define target: "success" = acquired OR ipo ──────────────────────────────
# The 'status' column contains: operating, acquired, closed, ipo
if 'status' in df.columns:
    df['success'] = df['status'].apply(
        lambda x: 1 if str(x).lower() in ['acquired', 'ipo'] else 0
    )
    print('Target distribution:')
    print(df['success'].value_counts())
else:
    print('Available columns:', df.columns.tolist())

## 📊 Step 5: Exploratory Data Analysis (EDA)

In [ ]:
# ── 5.1 Target class distribution ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

counts = df['success'].value_counts()
labels = ['Not Successful', 'Successful']
colors = ['#E07B54', '#4CAF81']

axes[0].pie(counts, labels=labels, autopct='%1.1f%%', colors=colors,
            startangle=140, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title('Startup Success Distribution', fontweight='bold')

axes[1].bar(labels, counts, color=colors, edgecolor='white', width=0.5)
axes[1].set_title('Count of Successful vs Not', fontweight='bold')
axes[1].set_ylabel('Count')
for i, v in enumerate(counts):
    axes[1].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── 5.2 Top 15 startup categories by success rate ────────────────────────────
cat_col = 'market' if 'market' in df.columns else 'category_list'

if cat_col in df.columns:
    cat_success = (
        df.groupby(cat_col)['success']
        .agg(['mean', 'count'])
        .query('count >= 20')
        .sort_values('mean', ascending=False)
        .head(15)
        .reset_index()
    )
    cat_success.columns = [cat_col, 'success_rate', 'count']

    fig, ax = plt.subplots(figsize=(14, 6))
    bars = ax.barh(cat_success[cat_col][::-1],
                   cat_success['success_rate'][::-1] * 100,
                   color=sns.color_palette('viridis', 15))
    ax.set_xlabel('Success Rate (%)')
    ax.set_title(f'Top 15 Markets by Startup Success Rate', fontweight='bold')
    for bar in bars:
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{bar.get_width():.1f}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── 5.3 Funding rounds analysis ──────────────────────────────────────────────
if 'funding_rounds' in df.columns:
    df['funding_rounds'] = pd.to_numeric(df['funding_rounds'], errors='coerce')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Distribution
    axes[0].hist(df['funding_rounds'].dropna(), bins=20, color='steelblue',
                 edgecolor='white', alpha=0.8)
    axes[0].set_title('Distribution of Funding Rounds', fontweight='bold')
    axes[0].set_xlabel('Number of Funding Rounds')
    axes[0].set_ylabel('Count')

    # Success by funding rounds
    round_success = df.groupby('funding_rounds')['success'].mean() * 100
    round_success = round_success[round_success.index <= 15]
    axes[1].plot(round_success.index, round_success.values,
                 marker='o', color='#4CAF81', linewidth=2.5, markersize=8)
    axes[1].fill_between(round_success.index, round_success.values,
                         alpha=0.15, color='#4CAF81')
    axes[1].set_title('Success Rate by # of Funding Rounds', fontweight='bold')
    axes[1].set_xlabel('Funding Rounds')
    axes[1].set_ylabel('Success Rate (%)')

    plt.tight_layout()
    plt.show()

In [ ]:
# ── 5.4 Total funding amount vs success ──────────────────────────────────────
fund_col = 'funding_total_usd'
if fund_col in df.columns:
    df[fund_col] = pd.to_numeric(
        df[fund_col].astype(str).str.replace(',', ''), errors='coerce'
    )
    df_fund = df[df[fund_col].between(1, 5e9)].copy()
    df_fund['log_funding'] = np.log1p(df_fund[fund_col])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Box plot
    df_fund.boxplot(column='log_funding', by='success', ax=axes[0],
                    patch_artist=True,
                    boxprops=dict(facecolor='lightblue'),
                    medianprops=dict(color='red', linewidth=2))
    axes[0].set_title('Log(Funding) by Success Class', fontweight='bold')
    axes[0].set_xlabel('Success (0=No, 1=Yes)')
    axes[0].set_ylabel('log(1 + Total Funding USD)')
    plt.sca(axes[0])
    plt.title('')

    # KDE
    for label, color in zip([0, 1], ['#E07B54', '#4CAF81']):
        subset = df_fund[df_fund['success'] == label]['log_funding']
        axes[1].hist(subset, bins=40, alpha=0.5, color=color, density=True,
                     label=['Not Successful', 'Successful'][label])
    axes[1].set_title('Funding Distribution by Outcome', fontweight='bold')
    axes[1].set_xlabel('log(1 + Total Funding USD)')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# ── 5.5 Top countries by startup count and success rate ──────────────────────
if 'country_code' in df.columns:
    country_stats = (
        df.groupby('country_code')['success']
        .agg(['mean', 'count'])
        .query('count >= 50')
        .sort_values('count', ascending=False)
        .head(20)
        .reset_index()
    )
    country_stats.columns = ['country', 'success_rate', 'total']

    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(country_stats))
    w = 0.4

    bars1 = ax.bar(x - w/2, country_stats['total'], w, label='Total Startups',
                   color='#5b8cdb', edgecolor='white')
    ax2 = ax.twinx()
    bars2 = ax2.bar(x + w/2, country_stats['success_rate'] * 100, w,
                    label='Success Rate %', color='#4CAF81', edgecolor='white')

    ax.set_xticks(x)
    ax.set_xticklabels(country_stats['country'], rotation=45, ha='right')
    ax.set_ylabel('Total Startups', color='#5b8cdb')
    ax2.set_ylabel('Success Rate (%)', color='#4CAF81')
    ax.set_title('Top 20 Countries: Startup Volume & Success Rate', fontweight='bold')

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

    plt.tight_layout()
    plt.show()

In [ ]:
# ── 5.6 Correlation heatmap ───────────────────────────────────────────────────
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'success' in num_cols:
    corr = df[num_cols].corr()

    mask = np.triu(np.ones_like(corr, dtype=bool))
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
                center=0, linewidths=0.5, ax=ax)
    ax.set_title('Feature Correlation Heatmap', fontweight='bold')
    plt.tight_layout()
    plt.show()

## 🔧 Step 6: Feature Engineering & Preprocessing

In [ ]:
# ── Select features ───────────────────────────────────────────────────────────
feature_candidates = [
    'funding_rounds', 'funding_total_usd', 'country_code', 'market',
    'has_VC', 'has_angel', 'has_roundA', 'has_roundB', 'has_roundC',
    'has_roundD', 'avg_participants', 'is_top500'
]

# Keep only columns that exist
features = [c for c in feature_candidates if c in df.columns]
print('Selected features:', features)

# Also add numeric cols not in the list
extra_num = [c for c in num_cols if c not in features + ['success']]
features = features + extra_num
features = list(dict.fromkeys(features))  # deduplicate, preserve order

Xy = df[features + ['success']].copy()
print(f'Working dataset: {Xy.shape}')

In [ ]:
# ── Encode categoricals, impute, scale ───────────────────────────────────────
le = LabelEncoder()
cat_feats = Xy.select_dtypes(include='object').columns.tolist()
for col in cat_feats:
    Xy[col] = le.fit_transform(Xy[col].astype(str))

X = Xy.drop(columns=['success'])
y = Xy['success']

# Impute missing
imp = SimpleImputer(strategy='median')
X_imp = pd.DataFrame(imp.fit_transform(X), columns=X.columns)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_imp, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Class balance (train): {y_train.value_counts().to_dict()}')

In [ ]:
# ── Handle class imbalance with SMOTE ────────────────────────────────────────
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_train, y_train)
print(f'After SMOTE: {y_res.value_counts().to_dict()}')

# Scale
scaler = StandardScaler()
X_res_sc = scaler.fit_transform(X_res)
X_test_sc = scaler.transform(X_test)

## 🤖 Step 7: Train Multiple Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=150, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=150, random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=150, random_state=42,
                                          eval_metric='logloss', verbosity=0)
}

results = {}

for name, model in models.items():
    # Use scaled data for LR, raw for tree-based
    if 'Logistic' in name:
        model.fit(X_res_sc, y_res)
        preds = model.predict(X_test_sc)
        proba = model.predict_proba(X_test_sc)[:, 1]
    else:
        model.fit(X_res, y_res)
        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, proba)
    results[name] = {'model': model, 'preds': preds, 'proba': proba,
                     'accuracy': acc, 'auc': auc}
    print(f'{name:25s} → Accuracy: {acc:.4f}  |  ROC-AUC: {auc:.4f}')

## 📈 Step 8: Model Evaluation & Visualization

In [ ]:
# ── 8.1 ROC Curves ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
palette = ['#3B82F6', '#EF4444', '#10B981', '#F59E0B']

for (name, res), color in zip(results.items(), palette):
    fpr, tpr, _ = roc_curve(y_test, res['proba'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})",
            linewidth=2.5, color=color)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── 8.2 Model Comparison Bar Chart ───────────────────────────────────────────
metrics_df = pd.DataFrame([
    {'Model': n, 'Accuracy': r['accuracy'], 'ROC-AUC': r['auc']}
    for n, r in results.items()
]).set_index('Model')

fig, ax = plt.subplots(figsize=(10, 5))
metrics_df.plot(kind='bar', ax=ax, width=0.6, edgecolor='white',
                color=['#3B82F6', '#10B981'])
ax.set_title('Model Performance Comparison', fontweight='bold', fontsize=14)
ax.set_ylabel('Score')
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha='right')
ax.set_ylim(0, 1.1)
ax.axhline(0.8, linestyle='--', color='red', alpha=0.4, label='0.8 Threshold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 8.3 Confusion Matrices ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()

for i, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res['preds'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['Not Successful', 'Successful'])
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
    axes[i].set_title(name, fontweight='bold')

plt.suptitle('Confusion Matrices', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 8.4 Feature Importance (Best Tree Model) ─────────────────────────────────
best_name = max(results, key=lambda k: results[k]['auc'])
best_model = results[best_name]['model']
print(f'Best model: {best_name} (AUC={results[best_name]["auc"]:.4f})')

if hasattr(best_model, 'feature_importances_'):
    fi = pd.Series(best_model.feature_importances_, index=X_test.columns)
    fi = fi.sort_values(ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(12, 6))
    fi.sort_values().plot(kind='barh', ax=ax,
                          color=sns.color_palette('crest', len(fi)))
    ax.set_title(f'Top 15 Feature Importances — {best_name}',
                 fontweight='bold', fontsize=14)
    ax.set_xlabel('Importance Score')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── 8.5 Classification Report (Best Model) ───────────────────────────────────
print(f'\n📊 Classification Report — {best_name}\n')
print(classification_report(y_test, results[best_name]['preds'],
                             target_names=['Not Successful', 'Successful']))

## 🔮 Step 9: Predict on New Startups

In [ ]:
# Predict on a hypothetical new startup
# Fill in values matching your feature columns

new_startup = pd.DataFrame([
    {col: 0 for col in X_test.columns}  # Baseline: all zeros
])

# Override specific known features (adjust to your actual columns)
if 'funding_rounds' in new_startup.columns:
    new_startup['funding_rounds'] = 3
if 'funding_total_usd' in new_startup.columns:
    new_startup['funding_total_usd'] = 5_000_000

# Impute then scale if needed
new_imp = pd.DataFrame(imp.transform(new_startup), columns=X_test.columns)

if 'Logistic' in best_name:
    new_sc = scaler.transform(new_imp)
    prob = best_model.predict_proba(new_sc)[0][1]
else:
    prob = best_model.predict_proba(new_imp)[0][1]

print(f'\n🚀 Predicted Success Probability: {prob:.2%}')
print(f'   Verdict: {"✅ Likely Successful" if prob >= 0.5 else "❌ Likely Not Successful"}')

---
## ✅ Summary

| Step | What we did |
|------|-------------|
| Data | Crunchbase VC investments dataset from Kaggle |
| Target | `success = 1` if startup was acquired or had IPO |
| EDA | Class balance, market success rates, funding analysis, country heatmap |
| Preprocessing | Drop high-NaN cols, label encode, median impute, SMOTE, StandardScaler |
| Models | Logistic Regression, Random Forest, Gradient Boosting, XGBoost |
| Evaluation | Accuracy, ROC-AUC, Confusion Matrix, Feature Importance |

> 💡 **Tips to improve further:** Try hyperparameter tuning with `GridSearchCV`, add NLP features from company descriptions, or use time-series features based on founding year.